In [1]:
# Imports
from rich.console import Console
from dotenv import load_dotenv
from anthropic import Anthropic
import json
load_dotenv(override=True)

True

In [2]:
# Claude instance
claude = Anthropic()

In [ ]:
def add_ingredients(ingredient: str, qty: int) -> list[dict]:
    ingredients = []
    ingredients.append({
        "ingredient": ingredient,
        "qty": qty,
    }
    )
    return ingredients

add_ingredients_tool = {
    "name": "add_ingredients",
    "description": "Adds one ingredient and its quantity to the current recipe's ingredient list. Use this whenever the user mentions an ingredient they have available or want included in the recipe, one call per ingredient. The 'ingredient' parameter should be the ingredient's name only (e.g. 'soy sauce', not 'a bottle of soy sauce'). The 'qty' parameter is a numeric amount without units — if the user gives a unit (e.g. '500 ml'), extract just the number. Do not use this tool to remove or update an existing ingredient; it only appends new entries.",
    "input_schema": {
        "type": "object",
        "properties": {
            "ingredient": {
                "type": "string",
                "description": "The name of the ingredient",
            },
            "qty": {
                "type": "integer",
                "description": "Ingredient's quantity"
            }
        },
        "required": ["ingredient", "qty"],
    }
}

In [4]:
tools = [add_ingredients_tool]

In [ ]:
add_ingredients("soja", 3)

In [ ]:
# Claude API call
messages = [{"role": "user", "content": "We have 500 ml of soja sauce"}]  # Conversation history — must start with a role:"user" entry per the Messages API.
while True:
    # Agent loop: one full request → response → (optional tool round) cycle per pass.
    response = claude.messages.create(
        messages=messages,
        model="claude-haiku-4-5-20251001",
        system="You are a chef, you are tasked for creating recipes",
        max_tokens=1023,
        timeout=59,
        tools=tools,
    )  # Sends the whole history; response.content is a list of content blocks (text and/or tool_use).
    messages.append({"role": "assistant", "content": response.content})  # Persists Claude's turn — required since tool_use blocks are only valid inside assistant messages.
    Console().print(response.content)
    # Extract tool_use block(s)
    if response.stop_reason != 'tool_use':
        break  # Exit condition: stop_reason != "tool_use" means Claude gave a final answer, not a tool request.
    tool_calls = [block for block in response.content if block.type == "tool_use"]
    # Filters this turn's content blocks down to just the tool_use blocks (Claude's tool requests).
    tool_results = []
    for block in tool_calls:
        # One iteration per requested tool_use block — a turn can request more than one.
        fn = globals().get(block.name)  # Dispatcher: resolves the tool_use block's name (string) to its Python function.
        result = fn(**block.input)  # Executes the tool, unpacking its input dict as keyword arguments.
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": str(result)
        }
        )  # Builds the tool_result content block Claude requires back: type, matching tool_use_id, content as string.
    messages.append({"role": "user", "content": tool_results})  # Sends the tool outcome(s) back as a single user message, immediately after the assistant's tool_use turn.
    Console().print(tool_results)

    #Console().print(response)

In [27]:
Console().print(response)

Message(
    id='msg_011CdWEtnnnnB7LFDQkPbTNE',
    content=[
        TextBlock(
            citations=None,
            text="Great! I've noted that you have 500 ml of soja sauce available. \n\nAre you looking to:\n1. **Add
this to your ingredient list** for a recipe you're planning?\n2. **Get recipe ideas** that use soja sauce?\n3. 
**Create a specific dish** with this ingredient?\n\nLet me know how you'd like to proceed, and I can help you build
a recipe! If you'd like me to add it to your ingredients list, just let me know and I'll do that right away.",
            type='text'
        )
    ],
    model='claude-haiku-4-5-20251001',
    role='assistant',
    stop_reason='end_turn',
    stop_sequence=None,
    type='message',
    usage=Usage(
        cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
        cache_creation_input_tokens=0,
        cache_read_input_tokens=0,
        input_tokens=617,
        output_tokens=118,
        server_tool_use=None,
        service_tier='standard',
        inference_geo='not_available'
    ),
    stop_details=None
)

In [ ]:
# Extract tool_use block(s)
if response.stop_reason == 'tool_use':
    tool_calls = [block for block in response.content if block.type == "tool_use"]
    tool_results = []
    for block in tool_calls:
        fn = globals().get(block.name)
        result = fn(**block.input)
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": str(result)
        }
        )
    print(tool_results)

[{'type': 'tool_result', 'tool_use_id': 'toolu_01FPfisSFGkBjwqycAF1r4vK', 'content': "[{'ingredient': 'soja sauce', 'qty': 500}, {'ingredient': 'soja sauce', 'qty': 500}, {'ingredient': 'soja sauce', 'qty': 500}, {'ingredient': 'soja sauce', 'qty': 500}, {'ingredient': 'soja sauce', 'qty': 500}, {'ingredient': 'soja sauce', 'qty': 500}, {'ingredient': 'soja sauce', 'qty': 500}, {'ingredient': 'soja sauce', 'qty': 500}]"}]


In [ ]:
answer = response.content[0].text
Console().print(answer)

In [ ]:
# 